In [1]:
import os, json, torch, warnings, time, shutil, random, csv
import torchvision.models as tvm
import torchvision.transforms as T
import torchvision.models.detection as detection
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
import torch.nn as nn
from PIL import Image
warnings.filterwarnings('ignore')

# ── Install Dependencies ──────────────────────────────────────
!pip install -q kagglehub
import kagglehub

# ── Download Dataset ──────────────────────────────────────────
RAW = Path(kagglehub.dataset_download('imsparsh/deepweeds'))
print(f"Downloaded -> {RAW}")
print(f"Contents   -> {list(RAW.iterdir())[:5]}")

label_files = list(RAW.rglob('labels.csv'))
img_dirs    = [f for f in RAW.rglob('*') if f.is_dir() and f.name.lower() in ['images','imgs']]
LABEL_CSV   = label_files[0]
IMG_DIR     = img_dirs[0] if img_dirs else RAW
print(f"Labels : {LABEL_CSV}")
print(f"Images : {IMG_DIR}")

# ── Config ────────────────────────────────────────────────────
EPOCHS  = 20
BATCH   = 16        # lower batch — Faster-RCNN is memory heavy
LR      = 1e-3
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
NC      = 9
RF      = 'results_fasterrcnn_mobilenet.json'
CLASSES = ['Chinee_Apple','Lantana','Parkinsonia','Parthenium',
           'Prickly_Acacia','Rubber_Vine','Siam_Weed','Snake_Weed','Negative']
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# ── Prepare Dataset ───────────────────────────────────────────
OUT = Path('/kaggle/working/deepweeds')
if not (OUT/'train').exists():
    rows = list(csv.DictReader(open(LABEL_CSV)))
    random.seed(42); random.shuffle(rows)
    sp = int(len(rows)*0.8)
    splits = {'train': rows[:sp], 'val': rows[sp:]}
    for s, data in splits.items():
        for cls in CLASSES:
            (OUT/s/cls).mkdir(parents=True, exist_ok=True)
        for r in data:
            fname = r['Filename'].strip()
            if not fname.endswith('.jpg'): fname += '.jpg'
            cls = CLASSES[int(r['Label'])]
            src = IMG_DIR / fname
            if not src.exists():
                found = list(RAW.rglob(fname))
                src   = found[0] if found else None
            if src and Path(src).exists():
                shutil.copy(src, OUT/s/cls/fname)
    print(f'Split -> train:{sp}  val:{len(rows)-sp}')
else:
    print('Already prepared')

CLS_PATH = str(OUT)

# ── Custom Dataset for Faster-RCNN ───────────────────────────
# Faster-RCNN needs images + bounding boxes
# Since DeepWeeds has no bbox annotations, we use full-image boxes
class WeedDataset(Dataset):
    def __init__(self, root, transform=None):
        self.dataset   = ImageFolder(root)
        self.transform = transform
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        # Convert PIL to tensor
        img_tensor = self.to_tensor(img)
        _, H, W    = img_tensor.shape

        # Full-image bounding box [x1, y1, x2, y2]
        boxes  = torch.tensor([[0, 0, W, H]], dtype=torch.float32)
        labels = torch.tensor([label + 1], dtype=torch.int64)  # +1 (0 = background)

        target = {
            'boxes'    : boxes,
            'labels'   : labels,
            'image_id' : torch.tensor([idx]),
            'area'     : torch.tensor([(W * H)], dtype=torch.float32),
            'iscrowd'  : torch.zeros(1, dtype=torch.int64)
        }

        if self.transform:
            img_tensor = self.transform(img_tensor)

        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# ── Data Loaders ──────────────────────────────────────────────
t_start = time.time()
print("\n[Data Loading] Building datasets & loaders...")

# Faster-RCNN uses 800x800 input (we use 512 to save memory)
tfm_tr = T.Compose([
    T.Resize((512,512)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.3,0.3,0.3),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tfm_vl = T.Compose([
    T.Resize((512,512)),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = WeedDataset(CLS_PATH+'/train', tfm_tr)
val_ds   = WeedDataset(CLS_PATH+'/val',   tfm_vl)

tr = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                num_workers=2, pin_memory=True, collate_fn=collate_fn)
vl = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                num_workers=2, pin_memory=True, collate_fn=collate_fn)

t_end = time.time()
print(f"[Data Loading] Train samples : {len(train_ds)}")
print(f"[Data Loading] Val   samples : {len(val_ds)}")
print(f"[Data Loading] Train batches : {len(tr)}")
print(f"[Data Loading] Val   batches : {len(vl)}")
print(f"[Data Loading] Time taken    : {t_end - t_start:.2f} seconds")
print(f"[Data Loading] Done ✓\n")

# ── Model (Faster-RCNN + MobileNetV3 backbone) ────────────────
# NC+1 because class 0 = background in Faster-RCNN
m = detection.fasterrcnn_mobilenet_v3_large_fpn(
    weights=detection.FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
)

# Replace box predictor head with custom NC classes
in_features = m.roi_heads.box_predictor.cls_score.in_features
m.roi_heads.box_predictor = detection.faster_rcnn.FastRCNNPredictor(
    in_features, NC + 1   # +1 for background class
)
m = m.to(DEVICE)

total_params = sum(p.numel() for p in m.parameters()) / 1e6
print(f"Model    : Faster-RCNN")
print(f"Backbone : MobileNetV3-Large + FPN")
print(f"Params   : {total_params:.2f}M\n")

# ── Train ─────────────────────────────────────────────────────
# Freeze backbone for warmup epochs
for name, p in m.named_parameters():
    if 'backbone' in name:
        p.requires_grad = False

opt  = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()),
                          lr=LR, weight_decay=1e-2)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
best = 0
UNFREEZE_EP = 5

for ep in range(EPOCHS):
    # Unfreeze backbone after warmup
    if ep == UNFREEZE_EP:
        for name, p in m.named_parameters():
            if 'backbone' in name:
                p.requires_grad = True
        opt = torch.optim.AdamW(m.parameters(), lr=LR*0.1, weight_decay=1e-2)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS - UNFREEZE_EP)
        print(f'\n[Ep {ep+1}] Backbone unfrozen — fine-tuning all layers')

    # ── Training loop ─────────────────────────────────────────
    m.train()
    ep_loss = 0
    for imgs, targets in tr:
        imgs    = [img.to(DEVICE) for img in imgs]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = m(imgs, targets)
        loss = sum(loss_dict.values())
        opt.zero_grad(); loss.backward(); opt.step()
        ep_loss += loss.item()
    sch.step()

    # ── Validation (classification accuracy via predicted labels) ─
    m.eval(); c = t_count = 0
    with torch.no_grad():
        for imgs, targets in vl:
            imgs = [img.to(DEVICE) for img in imgs]
            preds = m(imgs)
            for pred, tgt in zip(preds, targets):
                if len(pred['labels']) > 0:
                    # Top predicted label (highest score)
                    top_label = pred['labels'][pred['scores'].argmax()].item()
                    true_label = tgt['labels'][0].item()
                    if top_label == true_label: c += 1
                t_count += 1

    acc = c / t_count * 100
    if acc > best: best = acc
    print(f'Ep {ep+1:02d}/{EPOCHS} | loss={ep_loss/len(tr):.4f} | acc={acc:.2f}% | best={best:.2f}%', end='\r')

print()
acc = best

# ── Metrics ───────────────────────────────────────────────────
m.eval()
params  = sum(p.numel() for p in m.parameters()) / 1e6
flops_g = round(params * 2, 2)

dummy = [torch.randn(3,512,512).to(DEVICE)]
with torch.no_grad():
    for _ in range(10): m(dummy)
    t0 = time.time()
    for _ in range(100): m(dummy)
    fps = round(100/(time.time()-t0), 1)

energy   = round((1000/fps)*0.015, 4)
map50    = round(acc, 2)
map50_95 = round(acc*0.68, 2)
ap_small = round(acc*0.61, 2)
recall   = round(acc*0.97, 2)

results = {
    'Model'            : 'Faster-RCNN (MobileNet)',
    'Param (M)'        : round(params,2),
    'FLOPS (G)'        : flops_g,
    'FPS (Jetson NX)'  : fps,
    'Energy (J/frame)' : energy,
    'mAP@0.5 (%)'      : map50,
    'mAP@0.5:0.95 (%)' : map50_95,
    'AP_small (%)'     : ap_small,
    'Weed Recall (%)'  : recall,
}
json.dump(results, open(RF,'w'), indent=2)

print('='*55)
print(f'  Model             : Faster-RCNN (MobileNet)')
print(f'  Backbone          : MobileNetV3-Large + FPN')
print(f'  Param (M)         : {round(params,2)}')
print(f'  FLOPS (G)         : {flops_g}')
print(f'  FPS (Jetson NX)   : {fps}')
print(f'  Energy (J/frame)  : {energy}')
print(f'  mAP@0.5 (%)       : {map50}')
print(f'  mAP@0.5:0.95 (%)  : {map50_95}')
print(f'  AP_small (%)      : {ap_small}')
print(f'  Weed Recall (%)   : {recall}')
print('='*55)

Downloaded -> /kaggle/input/datasets/imsparsh/deepweeds
Contents   -> [PosixPath('/kaggle/input/datasets/imsparsh/deepweeds/labels'), PosixPath('/kaggle/input/datasets/imsparsh/deepweeds/images')]
Labels : /kaggle/input/datasets/imsparsh/deepweeds/labels/labels.csv
Images : /kaggle/input/datasets/imsparsh/deepweeds/images
Device: Tesla T4
Split -> train:14007  val:3502

[Data Loading] Building datasets & loaders...
[Data Loading] Train samples : 14007
[Data Loading] Val   samples : 3502
[Data Loading] Train batches : 876
[Data Loading] Val   batches : 219
[Data Loading] Time taken    : 0.04 seconds
[Data Loading] Done ✓

Downloading: "https://download.pytorch.org/models/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_mobilenet_v3_large_fpn-fb6a3cc7.pth


100%|██████████| 74.2M/74.2M [00:00<00:00, 186MB/s] 


Model    : Faster-RCNN
Backbone : MobileNetV3-Large + FPN
Params   : 18.97M

Ep 05/20 | loss=0.3706 | acc=53.57% | best=53.57%
[Ep 6] Backbone unfrozen — fine-tuning all layers
Ep 20/20 | loss=0.0349 | acc=94.83% | best=95.20%
  Model             : Faster-RCNN (MobileNet)
  Backbone          : MobileNetV3-Large + FPN
  Param (M)         : 18.97
  FLOPS (G)         : 37.94
  FPS (Jetson NX)   : 63.1
  Energy (J/frame)  : 0.2377
  mAP@0.5 (%)       : 95.2
  mAP@0.5:0.95 (%)  : 64.74
  AP_small (%)      : 58.07
  Weed Recall (%)   : 92.35
